# Species migration portfolio assignment

## Step 1: Get occurrence data from the Global Biodiversity Information Facility (GBIF)
For your species migration portfolio assignment, you will need to acquire GBIF data for a different species. I recommend choosing a terrestrial species, as this will work best with the ecoregions data used in the assignment.

## Set up
In the imports cell, we’ve included some packages that you will need. Add imports for additional packages that will help you to:
1. Work with reproducible file paths
2. Work with tabular data

In [ ]:
import time ### time processing steps
import zipfile ### read and extract .zip folders
from getpass import getpass ### handles sensitive inputs like passwords and API tokens
from glob import glob ### function for finding file paths
import requests ### make requests to websites and web APIs

### GBIF packages
import pygbif.occurrences as occ ### get GBIF occurrence data
import pygbif.species as species ### deals with species names

### import additional packages you'll need

### Step 1A: Make a GBIF account
You will need to make a free account with [GBIF](https://www.gbif.org/) to access their data. Keep track of your username, password, and the email address you sign up with, as you'll need all of this later!

You will need a (free) [GBIF account](https://www.gbif.org/) to complete this challenge. Then, run the following code to enter your credentials for the rest of your session.

This code is **interactive**, meaning that it will **ask you for a response**! The prompt can sometimes be
hard to see if you are using VSCode – it appears at the **top** of your editor window.

**Tips:**
1. If you need to save credentials across multiple sessions, you can consider loading them in from a file like a `.env`, but make sure to add it to .gitignore so you don’t commit your credentials to your repository!
2. If you accidentally enter your credentials wrong, you can set `reset = True` instead of `reset = False`.

**Warning**
Your email address **must** match the email you used to sign up for GBIF!

In [ ]:
### change this to reset = False after you run this chunk the first time
reset = True

####--------------------------------------####
#### DO NOT MODIFY THE REST OF THIS CODE! ####
####--------------------------------------####
### This code asks for your credentials and saves it for the rest of the session.
### NEVER put your credentials into your code!!!!
### GBIF needs a username, password, and email 

### request and store username
if (not ('GBIF_USER'  in os.environ)) or reset:
    os.environ['GBIF_USER'] = input('GBIF username:')

### securely request and store password
if (not ('GBIF_PWD'  in os.environ)) or reset:
    os.environ['GBIF_PWD'] = getpass('GBIF password:')
    
### request and store account email address
if (not ('GBIF_EMAIL'  in os.environ)) or reset:
    os.environ['GBIF_EMAIL'] = input('GBIF email:')

### Step 1B: Create a project folder and data directories
The code below will help you get started with making a folder for your data to live in:

In [ ]:
### create data directory
data_dir = os.path.join(

    ### point to your home directory
    pathlib.Path.home(),

    ### nest the data in the folder where your assignments live
    'my_assignment_folder',
    'data',
    'gbif_data')

### make the directory
os.makedirs(data_dir, exist_ok = True)

### define directory for the gbif data
gbif_dir = os.path.join(data_dir, 'gbif')

### make the directory
os.makedirs(gbif_dir, exist_ok = True)

### STEP 1C: Get the taxon key from GBIF

One of the tricky parts about getting occurrence data from GBIF is that species often have multiple names in different contexts. Luckily, GBIF also provides a Name Backbone service that will translate scientific and colloquial names into unique identifiers. GBIF calls these identifiers **taxon keys**:

1. Put the species name into the correct location in the code below.
2. Examine the object you get back from the species query. What part of it do you think might be the taxon key
3. Extract and save the taxon key.

In [ ]:
### define the species name you will be using
backbone = species.name_backbone(name = )

### check it out
backbone

Depending on the output from "backbone," you may need to modify the column in the code below:

In [ ]:
### pull out the species key
species_key = backbone['acceptedUsageKey']
species_key

### Step 1D: Download data from GBIF

Downloading GBIF data is a multi-step process. However, we’ve provided you with a chunk of code that handles the API communications and caches the download. You’ll still need to customize your search.

Submit a request to GBIF using the following steps:
1. Replace `csv_file_pattern` with a string that will match **any** `.csv` file when used in the `.rglob()` method. HINT: The character `*` represents any number of any values except the file separator (e.g. `/` on UNIX systems).
2. Add parameters to the GBIF download function, `occ.download()` to limit your query to:
    - observations of 
    - from 
    - with spatial coordinates.
3. Then, run the download. This can take a few minutes. You can check your downloads by logging on to the [GBIF website](https://www.gbif.org/user/download").

In [ ]:
### set file name for download
gbif_pattern = os.path.join(gbif_dir, '*.csv')

### set the code to only download your data the first time you run this chunk (since it can be slow)
if not glob(gbif_pattern):

    ### only submit one request
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:

        ### submit a request to GBIF
        gbif_query = occ.download([

            ### add your species key
            f"speciesKey = {species_key}",

            ### filter out results without geographic coordinates
            'hasCoordinate = ',

            ### choose a year to include
            f'year = ',
        ])

        ### grab the first result
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]

    ### wait for the download to build
    download_key = os.environ['GBIF_DOWNLOAD_KEY']

    ### use the occurrence command module in pygbif to get the metadata
    wait = occ.download_meta(download_key)['status']

    ### check if the status of the download = "SUCCEEDED"
    while not wait=='SUCCEEDED':
        wait = occ.download_meta(download_key)['status']

        ### wait before re-querying the API
        time.sleep(5)

    ### download GBIF data when it's ready
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'], 
        path = gbif_dir)

    ### unzip GBIF data using the zipfile package
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path = gbif_dir)

### find the extracted .csv file path (take the first result)
gbif_path = glob(gbif_pattern)[0]

You might notice that the GBIF data filename isn’t very **descriptive**. At this point, you may want to clean up your data directory so that you know what the file is later on!

1. Replace ‘your-gbif-filename’ with a descriptive name.
2. Run the cell.
3. Check your data folder. Is it organized the way you want?

In [ ]:
### give the download a descriptive name
gbif_path = project.project_dir / 'your-gbif-filename'

### move file to descriptive path
shutil.move(original_gbif_path, gbif_path)

### Step 1E: Load the GBIF data into Python

Just like you did when wrangling your data from the data subset, you’ll need to load your GBIF data and convert it to a GeoDataFrame.

In [ ]:
### load the GBIF data

### convert to GeoDataFrame

### check results
gbif_gdf.total_bounds

## Wrap up Portfolio Step 1

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [15]:
%store var1 var2

Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!